EfficientNet-V2B3 — Keras Tuner Hyperparameter Optimization
=============================================================
Purpose: Systematically tune both ARCHITECTURAL and TRAINING hyperparameters
of the best-performing architecture (EfficientNet-V2B3) using Bayesian Optimization.

Fixed decisions (determined by manual iterative experimentation):
  - Backbone: EfficientNet-V2B3 (ImageNet pretrained)
  - Loss: MAE (ordinal regression)
  - Output: Dense(1, linear)
  - Training: Two-phase (frozen head warm-up → full fine-tune)
  - Data: Balanced training set, no CLAHE, no augmentation
  - Preprocessing: EfficientNetV2 native (include_preprocessing=True)

Tuned hyperparameters:
  ARCHITECTURAL:
  - Dense units (head width): [128, 256, 512]
  - Number of Dense layers (head depth): [1, 2]
  REGULARIZATION:
  - Dropout rate: [0.3, 0.4, 0.5, 0.6, 0.7]
  - L2 regularization: [1e-4, 2e-4, 5e-4]
  TRAINING:
  - Phase 1 learning rate: [5e-4, 1e-3, 2e-3]




In [43]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import json
import keras
import numpy as np
import pandas as pd

import tensorflow as tf
from pathlib import Path
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, mean_absolute_error,
    cohen_kappa_score, roc_auc_score, classification_report
)
from sklearn.preprocessing import label_binarize

# Reproducibility 
SEED = 12049
tf.keras.utils.set_random_seed(SEED)

# GPU Setup 
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except Exception as exc:
            print(f"Could not enable memory growth for {gpu}: {exc}")

# Configuration settings
IMG_SIZE = 224
BATCH_SIZE = 16
NUM_CLASSES = 5
CLASS_NAMES = ['Healthy', 'Doubtful', 'Minimal', 'Moderate', 'Severe']
IMAGE_EXTENSIONS = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'}
MODEL_NAME = 'EfficientNetV2B3_KerasTuner'


# Tuner Configuration
MAX_TRIALS = 30          # Number of hyperparameter combinations to try
PHASE1_EPOCHS = 15       # Epochs per trial (Phase 1 only — frozen backbone)
PHASE1_PATIENCE = 5      # Early stopping patience during search
PHASE2_EPOCHS = 35       # Epochs for final fine-tuning (Phase 2)
PHASE2_PATIENCE = 7      # Early stopping patience for Phase 2


# Output Directories 
OUTPUT_DIR = Path('tuner_results')
MODELS_DIR = Path('models')
REPORTS_DIR = Path('reports')
for d in (OUTPUT_DIR, MODELS_DIR, REPORTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

In [42]:
print(f"TensorFlow: {tf.__version__}")
print(f"GPUs: {gpus}")

TensorFlow: 2.21.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 1.0 DATA LOADING

In [14]:
def find_dataset_dir() -> Path:
    """Locate the dataset directory by searching upward from cwd."""
    search_roots = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    candidate_suffixes = [
        'ml_workflow/data/knee_osteoarthritis',
        'Final_year_project/ml_workflow/data/knee_osteoarthritis',
        'data/knee_osteoarthritis',
    ]
    for root in search_roots:
        for suffix in candidate_suffixes:
            candidate = root / suffix
            if candidate.exists() and (candidate / 'train_balanced').exists():
                return candidate
    raise FileNotFoundError(
        "Cannot locate dataset directory. "
        "Expected structure: .../data/knee_osteoarthritis/train_balanced/"
    )

In [15]:
BASE_DIR = find_dataset_dir()
TRAIN_PATH = BASE_DIR / 'train_balanced'
VALID_PATH = BASE_DIR / 'val'
TEST_PATH = BASE_DIR / 'test'

print(f"\nDataset found at: {BASE_DIR}")
print(f"  Train: {TRAIN_PATH}")
print(f"  Valid: {VALID_PATH}")
print(f"  Test:  {TEST_PATH}")


Dataset found at: /workspaces/fyp_experiment/Final_year_project/ml_workflow/data/knee_osteoarthritis
  Train: /workspaces/fyp_experiment/Final_year_project/ml_workflow/data/knee_osteoarthritis/train_balanced
  Valid: /workspaces/fyp_experiment/Final_year_project/ml_workflow/data/knee_osteoarthritis/val
  Test:  /workspaces/fyp_experiment/Final_year_project/ml_workflow/data/knee_osteoarthritis/test


In [16]:
def get_split_samples(split_path):
    """Load all image paths and labels from a split directory."""
    samples = []
    for class_idx in range(NUM_CLASSES):
        class_dir = split_path / str(class_idx)
        if class_dir.exists():
            for fname in sorted(os.listdir(class_dir)):
                if Path(fname).suffix.lower() in IMAGE_EXTENSIONS:
                    samples.append((str(class_dir / fname), class_idx))
    return samples

def load_and_preprocess(path, label):
    """Load image from path and return as float32 [0, 255] for EfficientNetV2."""
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.cast(img, tf.float32)  # EfficientNetV2 include_preprocessing handles scaling
    return img, tf.cast(label, tf.float32)

def make_dataset(samples, shuffle=False):
    """Create a tf.data.Dataset from samples."""
    paths = [s[0] for s in samples]
    labels = [s[1] for s in samples]
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(len(samples), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

In [17]:
# Load data
print("\nLoading data splits...")
train_samples = get_split_samples(TRAIN_PATH)
valid_samples = get_split_samples(VALID_PATH)
test_samples = get_split_samples(TEST_PATH)

print(f"  Train: {len(train_samples)} samples")
print(f"  Valid: {len(valid_samples)} samples")
print(f"  Test:  {len(test_samples)} samples")

train_ds = make_dataset(train_samples, shuffle=True)
valid_ds = make_dataset(valid_samples, shuffle=False)
test_ds = make_dataset(test_samples, shuffle=False)


Loading data splits...
  Train: 11226 samples
  Valid: 826 samples
  Test:  1656 samples


In [18]:
# Compute class weights (same as C4 — smoothed)
train_labels = np.array([s[1] for s in train_samples])
raw_weights = compute_class_weight('balanced', classes=np.arange(NUM_CLASSES), y=train_labels)
smoothed_class_weights = {int(i): float(np.sqrt(w)) for i, w in enumerate(raw_weights)}
print(f"  Smoothed class weights: {smoothed_class_weights}")

  Smoothed class weights: {0: 0.9998218897483424, 1: 1.0000445424378297, 2: 1.0000445424378297, 3: 1.0000445424378297, 4: 1.0000445424378297}


## 2.0 KERAS TUNER — Model Building

In [19]:
import keras_tuner as kt

In [20]:
def build_model(hp):
    dropout_rate  = hp.Choice('dropout', values=[0.3, 0.4, 0.5, 0.6, 0.7])
    l2_value      = hp.Choice('l2', values=[1e-4, 2e-4, 5e-4])
    learning_rate = hp.Choice('learning_rate', values=[5e-4, 1e-3, 2e-3])
    dense_units   = hp.Choice('dense_units', values=[128, 256, 512])
    num_layers    = hp.Choice('num_dense_layers', values=[1, 2])

    base_model = keras.applications.EfficientNetV2B3(
        include_top=False,
        include_preprocessing=True,
        weights='imagenet',
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
    )
    base_model.trainable = False

    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = base_model(inputs, training=False)
    x = keras.layers.GlobalAveragePooling2D()(x)
    x = keras.layers.BatchNormalization()(x)

    # First Dense layer (always present)
    x = keras.layers.Dense(
        dense_units, activation='relu',
        kernel_regularizer=keras.regularizers.l2(l2_value)
    )(x)
    x = keras.layers.Dropout(dropout_rate)(x)

    # Optional second Dense layer (architectural tuning)
    if num_layers == 2:
        x = keras.layers.Dense(
            dense_units // 2, activation='relu',
            kernel_regularizer=keras.regularizers.l2(l2_value)
        )(x)
        x = keras.layers.Dropout(dropout_rate)(x)

    outputs = keras.layers.Dense(1, activation='linear')(x)

    model = keras.Model(inputs, outputs, name=MODEL_NAME)

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='mae',
        metrics=['mae'],
    )
    return model


## 3.0 RUN KERAS TUNER (Bayesian Optimization)
Tuning is done on VALIDATION SET only — test set is NOT touched.


In [21]:
print("\n" + "=" * 70)
print("  KERAS TUNER — Bayesian Optimization")
print(f"  Search space: dense_units × num_dense_layers × dropout × L2 × learning_rate")
print(f"  Max trials: {MAX_TRIALS}")
print(f"  Epochs per trial: {PHASE1_EPOCHS} (Phase 1 only, frozen backbone)")
print(f"  Objective: minimize val_mae")
print("=" * 70 + "\n")

tuner = kt.BayesianOptimization(
    build_model,
    objective='val_mae',
    max_trials=MAX_TRIALS,
    seed=SEED,
    directory=str(OUTPUT_DIR),
    project_name='efficientnet_v2b3_bayesian',
    overwrite=True,
)

# Display search space
tuner.search_space_summary()


  KERAS TUNER — Bayesian Optimization
  Search space: dropout × L2 × learning_rate
  Max trials: 15
  Epochs per trial: 15 (Phase 1 only, frozen backbone)
  Objective: minimize val_mae

Search space summary
Default search space size: 3
dropout (Choice)
{'default': 0.3, 'conditions': [], 'values': [0.3, 0.4, 0.5, 0.6, 0.7], 'ordered': True}
l2 (Choice)
{'default': 0.0001, 'conditions': [], 'values': [0.0001, 0.0002, 0.0005], 'ordered': True}
learning_rate (Choice)
{'default': 0.0005, 'conditions': [], 'values': [0.0005, 0.001, 0.002], 'ordered': True}


In [22]:
# Run the search — VALIDATION SET only, test set NOT used
tuner.search(
    train_ds,
    validation_data=valid_ds,
    epochs=PHASE1_EPOCHS,
    class_weight=smoothed_class_weights,
    callbacks=[
        keras.callbacks.EarlyStopping(
            monitor='val_mae',
            patience=PHASE1_PATIENCE,
            restore_best_weights=True,
            verbose=1,
        ),
    ],
    verbose=1,
)


Trial 15 Complete [00h 03m 17s]
val_mae: 0.7296644449234009

Best val_mae So Far: 0.6997190117835999
Total elapsed time: 01h 15m 05s


In [23]:
best_hp = tuner.get_best_hyperparameters(num_trials=1)[0]

print("\n" + "=" * 70)
print("  BEST HYPERPARAMETERS FOUND")
print("=" * 70)
print(f"  Dense Units:       {best_hp.get('dense_units')}")
print(f"  Num Dense Layers:  {best_hp.get('num_dense_layers')}")
print(f"  Dropout:           {best_hp.get('dropout')}")
print(f"  L2:                {best_hp.get('l2')}")
print(f"  Learning Rate:     {best_hp.get('learning_rate')}")
print("=" * 70 + "\n")


  BEST HYPERPARAMETERS FOUND
  Dropout:       0.3
  L2:            0.0001
  Learning Rate: 0.0005



In [24]:
# Save best hyperparameters
best_hp_dict = {
    'dense_units': best_hp.get('dense_units'),
    'num_dense_layers': best_hp.get('num_dense_layers'),
    'dropout': best_hp.get('dropout'),
    'l2': best_hp.get('l2'),
    'learning_rate': best_hp.get('learning_rate'),
}
with open(REPORTS_DIR / 'best_hyperparameters.json', 'w') as f:
    json.dump(best_hp_dict, f, indent=2)

# Show all trial results
tuner.results_summary(num_trials=MAX_TRIALS)

Results summary
Results in tuner_results/efficientnet_v2b3_bayesian
Showing 15 best trials
Objective(name="val_mae", direction="min")

Trial 10 summary
Hyperparameters:
dropout: 0.3
l2: 0.0001
learning_rate: 0.0005
Score: 0.6997190117835999

Trial 02 summary
Hyperparameters:
dropout: 0.5
l2: 0.0002
learning_rate: 0.001
Score: 0.700158953666687

Trial 03 summary
Hyperparameters:
dropout: 0.3
l2: 0.0001
learning_rate: 0.0005
Score: 0.701468825340271

Trial 12 summary
Hyperparameters:
dropout: 0.3
l2: 0.0001
learning_rate: 0.0005
Score: 0.7045164704322815

Trial 11 summary
Hyperparameters:
dropout: 0.3
l2: 0.0001
learning_rate: 0.0005
Score: 0.7074522376060486

Trial 01 summary
Hyperparameters:
dropout: 0.7
l2: 0.0002
learning_rate: 0.0005
Score: 0.7099871635437012

Trial 06 summary
Hyperparameters:
dropout: 0.4
l2: 0.0002
learning_rate: 0.001
Score: 0.7161237597465515

Trial 09 summary
Hyperparameters:
dropout: 0.5
l2: 0.0002
learning_rate: 0.001
Score: 0.7164759635925293

Trial 00 summa


### 3.1 TRAIN FINAL MODEL WITH BEST HYPERPARAMETERS (Full Two-Phase)


In [25]:
print("\n" + "=" * 70)
print("  TRAINING FINAL MODEL (Two-Phase Fine-Tuning)")
print("  Phase 1: Frozen backbone (head warm-up)")
print("  Phase 2: Full backbone fine-tuning at lower LR")
print("=" * 70 + "\n")


  TRAINING FINAL MODEL (Two-Phase Fine-Tuning)
  Phase 1: Frozen backbone (head warm-up)
  Phase 2: Full backbone fine-tuning at lower LR



In [26]:
# Phase 1: Build model with best hyperparameters, frozen backbone 
final_model = build_model(best_hp)
final_model.summary()

Model: "EfficientNetV2B3_KerasTuner"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetv2-b3 (Functional)  │ (None, 7, 7, 1536)     │    12,930,622 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1536)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 1536)           │         6,144 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 256)            │       393,472 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │           257 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 13,330,495 (50.85 MB)

 Trainable params: 396,801 (1.51 MB)

 Non-trainable params: 12,933,694 (49.34 MB)

In [27]:
print("\n--- Phase 1: Head Warm-up (Frozen Backbone) ---")
history_phase1 = final_model.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=PHASE1_EPOCHS,
    class_weight=smoothed_class_weights,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor='val_mae', patience=PHASE1_PATIENCE,
            restore_best_weights=True, verbose=1
        ),
        tf.keras.callbacks.ModelCheckpoint(
            str(MODELS_DIR / 'best_phase1.keras'),
            monitor='val_mae', save_best_only=True, verbose=1
        ),
    ],
    verbose=1,
)

actual_phase1_epochs = len(history_phase1.history['loss'])
print(f"\nPhase 1 complete: {actual_phase1_epochs} epochs")


--- Phase 1: Head Warm-up (Frozen Backbone) ---
Epoch 1/15


I0000 00:00:1782225419.291098    9471 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_1365824__.396


699/702 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - loss: 1.3771 - mae: 1.3328

I0000 00:00:1782225454.453341    9471 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_1365824__.396


702/702 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - loss: 1.3764 - mae: 1.3320
Epoch 1: val_mae improved from inf to 0.80729, saving model to models/best_phase1.keras
702/702 ━━━━━━━━━━━━━━━━━━━━ 84s 72ms/step - loss: 1.1997 - mae: 1.1552 - val_loss: 0.8519 - val_mae: 0.8073
Epoch 2/15
702/702 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - loss: 0.9292 - mae: 0.8846
Epoch 2: val_mae did not improve from 0.80729
702/702 ━━━━━━━━━━━━━━━━━━━━ 16s 23ms/step - loss: 0.8973 - mae: 0.8527 - val_loss: 0.8621 - val_mae: 0.8176
Epoch 3/15
702/702 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - loss: 0.8280 - mae: 0.7834
Epoch 3: val_mae improved from 0.80729 to 0.74554, saving model to models/best_phase1.keras
702/702 ━━━━━━━━━━━━━━━━━━━━ 17s 24ms/step - loss: 0.8174 - mae: 0.7729 - val_loss: 0.7902 - val_mae: 0.7455
Epoch 4/15
701/702 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - loss: 0.7976 - mae: 0.7529
Epoch 4: val_mae improved from 0.74554 to 0.73197, saving model to models/best_phase1.keras
702/702 ━━━━━━━━━━━━━━━━━━━━ 17s 25ms/

In [28]:
# Phase 2: Unfreeze backbone, fine-tune at lower LR 
print("\n--- Phase 2: Full Fine-Tuning (Unfrozen Backbone) ---")


--- Phase 2: Full Fine-Tuning (Unfrozen Backbone) ---


In [30]:
# Unfreeze the EfficientNet backbone
final_model.layers[1].trainable = True

# Recompile with lower LR and AdamW 
final_model.compile(
    optimizer=keras.optimizers.AdamW(
        learning_rate=1e-5,
        weight_decay=best_hp.get('l2'),
    ),
    loss='mae',
    metrics=['mae'],
)


In [31]:
phase2_start = actual_phase1_epochs
phase2_end = actual_phase1_epochs + PHASE2_EPOCHS

In [32]:
history_phase2 = final_model.fit(
    train_ds,
    validation_data=valid_ds,
    initial_epoch=phase2_start,
    epochs=phase2_end,
    class_weight=smoothed_class_weights,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor='val_mae', patience=PHASE2_PATIENCE,
            restore_best_weights=True, verbose=1
        ),
        tf.keras.callbacks.ModelCheckpoint(
            str(MODELS_DIR / 'best_final.keras'),
            monitor='val_mae', save_best_only=True, verbose=1
        ),
    ],
    verbose=1,
)

actual_phase2_epochs = len(history_phase2.history['loss'])
total_epochs = actual_phase1_epochs + actual_phase2_epochs
print(f"\nPhase 2 complete: {actual_phase2_epochs} epochs")
print(f"Total epochs: {total_epochs}")

# Save final model
final_model.save(MODELS_DIR / 'final_tuned_model.keras')
print(f"Final model saved to {MODELS_DIR / 'final_tuned_model.keras'}")

Epoch 13/47


I0000 00:00:1782225837.099643    9474 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_1501520__.748
I0000 00:00:1782225867.832983   63882 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'input_reduce_fusion_1', 8 bytes spill stores, 8 bytes spill loads

I0000 00:00:1782225898.227242    9474 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'input_reduce_fusion_1', 8 bytes spill stores, 8 bytes spill loads



701/702 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step - loss: 0.9797 - mae: 0.9339

I0000 00:00:1782225960.562427    9473 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_1501520__.748


702/702 ━━━━━━━━━━━━━━━━━━━━ 0s 160ms/step - loss: 0.9796 - mae: 0.9338
Epoch 13: val_mae improved from inf to 0.77463, saving model to models/best_final.keras
702/702 ━━━━━━━━━━━━━━━━━━━━ 277s 189ms/step - loss: 0.8962 - mae: 0.8505 - val_loss: 0.8203 - val_mae: 0.7746
Epoch 14/47
702/702 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step - loss: 0.8002 - mae: 0.7546
Epoch 14: val_mae improved from 0.77463 to 0.73988, saving model to models/best_final.keras
702/702 ━━━━━━━━━━━━━━━━━━━━ 53s 76ms/step - loss: 0.7822 - mae: 0.7366 - val_loss: 0.7854 - val_mae: 0.7399
Epoch 15/47
701/702 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step - loss: 0.7391 - mae: 0.6936
Epoch 15: val_mae improved from 0.73988 to 0.70495, saving model to models/best_final.keras
702/702 ━━━━━━━━━━━━━━━━━━━━ 52s 74ms/step - loss: 0.7269 - mae: 0.6815 - val_loss: 0.7503 - val_mae: 0.7050
Epoch 16/47
701/702 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step - loss: 0.6895 - mae: 0.6442
Epoch 16: val_mae improved from 0.70495 to 0.68352, saving model to models/bes


### 4.0 FINAL EVALUATION ON TEST SET 
This is the ONLY time the test set is used — no tuning decisions are based
on test set performance. This prevents test-set leakage.


In [34]:
# Get predictions
test_labels = np.array([s[1] for s in test_samples])
predictions_raw = final_model.predict(test_ds, verbose=1).flatten()

# Convert ordinal predictions to class labels 
predictions_rounded = np.clip(np.round(predictions_raw), 0, NUM_CLASSES - 1).astype(int)

104/104 ━━━━━━━━━━━━━━━━━━━━ 35s 173ms/step


In [35]:
# Metrics 
accuracy = accuracy_score(test_labels, predictions_rounded)
balanced_acc = balanced_accuracy_score(test_labels, predictions_rounded)
mae = mean_absolute_error(test_labels, predictions_raw)
qwk = cohen_kappa_score(test_labels, predictions_rounded, weights='quadratic')

In [41]:
# For ROC AUC — need probability-like scores per class
# Convert raw ordinal scores to pseudo-probabilities using distance
def ordinal_to_probs(raw_scores, n_classes=5):
    """Convert raw ordinal scores to pseudo-probabilities for AUC."""
    probs = np.zeros((len(raw_scores), n_classes))
    for i, score in enumerate(raw_scores):
        for c in range(n_classes):
            probs[i, c] = np.exp(-abs(score - c))
        probs[i] /= probs[i].sum()
    return probs

pred_probs = ordinal_to_probs(predictions_raw)
y_true_onehot = label_binarize(test_labels, classes=np.arange(NUM_CLASSES))

try:
    macro_auc = roc_auc_score(y_true_onehot, pred_probs, average='macro', multi_class='ovr')
    weighted_auc = roc_auc_score(y_true_onehot, pred_probs, average='weighted', multi_class='ovr')
except Exception:
    macro_auc = float('nan')
    weighted_auc = float('nan')

# Print Results 
sep = '─' * 70
print(f"\n{sep}")
print(f"  COMPREHENSIVE EVALUATION METRICS — {MODEL_NAME}")
print(f"  Split: Test (FINAL — used once)")
print(f"{sep}")
print(f"  {'Overall Accuracy':<40} {accuracy:>10.4f}")
print(f"  {'Balanced Accuracy':<40} {balanced_acc:>10.4f}")
print(f"  {'Mean Absolute Error (MAE)':<40} {mae:>10.4f}")
print(f"  {'Macro ROC AUC':<40} {macro_auc:>10.4f}")
print(f"  {'Weighted ROC AUC':<40} {weighted_auc:>10.4f}")
print(f"  {'Quadratic Weighted Kappa (QWK)':<40} {qwk:>10.4f}")
print(f"{sep}")


──────────────────────────────────────────────────────────────────────
  COMPREHENSIVE EVALUATION METRICS — EfficientNetV2B3_KerasTuner
  Split: Test (FINAL — used once)
──────────────────────────────────────────────────────────────────────
  Overall Accuracy                             0.6171
  Balanced Accuracy                            0.6150
  Mean Absolute Error (MAE)                    0.4999
  Macro ROC AUC                                0.8616
  Weighted ROC AUC                             0.8330
  Quadratic Weighted Kappa (QWK)               0.8024
──────────────────────────────────────────────────────────────────────


In [39]:
print(f"\n  Best Hyperparameters Used:")
print(f"    Dense Units:       {best_hp.get('dense_units')}")
print(f"    Num Dense Layers:  {best_hp.get('num_dense_layers')}")
print(f"    Dropout:           {best_hp.get('dropout')}")
print(f"    L2:                {best_hp.get('l2')}")
print(f"    Learning Rate:     {best_hp.get('learning_rate')}")
print(f"{sep}")


  Best Hyperparameters Used:
    Dropout:       0.3
    L2:            0.0001
    Learning Rate: 0.0005
──────────────────────────────────────────────────────────────────────


In [46]:
import pandas as pd


In [49]:
# Load C4 metrics from its actual results file (not hardcoded)
c4_metrics_path = Path('/workspaces/fyp_experiment/Final_year_project/ml_workflow/training/knee_osteoarthritis/EfficientNet_V2B3/EfficientNet_V2B3_C4/reports/metrics_test.csv')
c4_df = pd.read_csv(c4_metrics_path)
c4_qwk = c4_df['Quadratic Weighted Kappa'].values[0]
c4_mae = c4_df['MAE'].values[0]
c4_acc = c4_df['Overall Accuracy'].values[0]

# Comparison with C4 
print(f"\n  COMPARISON WITH MANUAL C4 CONFIGURATION")
print(f"  {'Metric':<30} {'C4 (Manual)':<15} {'Tuner-Optimized':<15} {'Delta':<10}")
print(f"  {'-'*70}")

print(f"  {'QWK':<30} {c4_qwk:<15.4f} {qwk:<15.4f} {qwk - c4_qwk:<+10.4f}")
print(f"  {'MAE':<30} {c4_mae:<15.4f} {mae:<15.4f} {mae - c4_mae:<+10.4f}")
print(f"  {'Accuracy':<30} {c4_acc:<15.4f} {accuracy:<15.4f} {accuracy - c4_acc:<+10.4f}")
print(f"{sep}\n")

# ── Classification Report 
print("Classification Report:")
print(classification_report(test_labels, predictions_rounded,
                            target_names=CLASS_NAMES, digits=4))


  COMPARISON WITH MANUAL C4 CONFIGURATION
  Metric                         C4 (Manual)     Tuner-Optimized Delta     
  ----------------------------------------------------------------------
  QWK                            0.8092          0.8024          -0.0068   
  MAE                            0.4130          0.4999          +0.0869   
  Accuracy                       0.6576          0.6171          -0.0405   
──────────────────────────────────────────────────────────────────────

Classification Report:
              precision    recall  f1-score   support

     Healthy     0.6940    0.8341    0.7576       639
    Doubtful     0.2927    0.3243    0.3077       296
     Minimal     0.6631    0.4183    0.5130       447
    Moderate     0.7273    0.7534    0.7401       223
      Severe     0.8085    0.7451    0.7755        51

    accuracy                         0.6171      1656
   macro avg     0.6371    0.6150    0.6188      1656
weighted avg     0.6219    0.6171    0.6094      16

In [50]:
# ── Save Results ──────────────────────────────────────────────────────────────
results = {
    'model': MODEL_NAME,
    'best_hyperparameters': best_hp_dict,
    'test_metrics': {
        'accuracy': float(accuracy),
        'balanced_accuracy': float(balanced_acc),
        'mae': float(mae),
        'macro_roc_auc': float(macro_auc),
        'weighted_roc_auc': float(weighted_auc),
        'qwk': float(qwk),
    },
    'comparison_with_c4': {
        'c4_qwk': c4_qwk,
        'tuner_qwk': float(qwk),
        'delta_qwk': float(qwk - c4_qwk),
        'c4_mae': c4_mae,
        'tuner_mae': float(mae),
        'delta_mae': float(mae - c4_mae),
    },
    'training_info': {
        'total_trials': MAX_TRIALS,
        'phase1_epochs': actual_phase1_epochs,
        'phase2_epochs': actual_phase2_epochs,
        'total_epochs': total_epochs,
    },
    'note': (
        'Tuning was performed on validation set only. '
        'Test set was used ONCE at the end for final evaluation. '
        'No test-set leakage.'
    ),
}

with open(REPORTS_DIR / 'tuner_evaluation_results.json', 'w') as f:
    json.dump(results, f, indent=2)


print(f"\nResults saved to {REPORTS_DIR / 'tuner_evaluation_results.json'}")
print("\n✓ Keras Tuner hyperparameter optimization complete.")


Results saved to reports/tuner_evaluation_results.json

✓ Keras Tuner hyperparameter optimization complete.
